# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q linearmodels pandas numpy matplotlib seaborn statsmodels
    print('Packages installed.')


# Causal Inference: Instrumental Variables (IV) In-Depth

One of the biggest challenges in social science is **Endogeneity** (Omitted Variable Bias). This happens when there is a hidden variable that affects both your treatment (cause) and your outcome (effect).

For example, let's say we want to study the effect of **Education** on **Income**.
- Problem: **Ability** (unobserved) affects both Education and Income.
- Result: Naive regression overestimates the return to education.

**Instrumental Variables (IV)** allow us to isolate the causal effect by using a third variable (Z) that affects Education (X) but has *no direct effect* on Income (Y) except through Education.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
print("Libraries loaded.")

## 1. Simulating the Endogeneity Problem

We will generate data where:
- $U$ is "Ability" (Unobserved Confounder).
- $Z$ is "Distance to College" (Instrument). It affects Education but not Ability.
- $X$ is "Years of Education". Depends on Ability ($U$) and Distance ($Z$).
- $Y$ is "Income". Depends on Education ($X$) and Ability ($U$).

**True Causal Effect of X on Y = 10**.

In [ ]:
n = 1000

# 1. Unobserved Confounder (Ability)
U = np.random.normal(0, 1, n)

# 2. Instrument (Distance to College)
# Randomly distributed, not correlated with Ability
Z = np.random.normal(0, 1, n)

# 3. Treatment (Education)
# Depends on Z (Instrument) and U (Confounder)
# Living closer (lower Z) -> More education
# Higher Ability (higher U) -> More education
X = 12 + 0.5 * U - 0.5 * Z + np.random.normal(0, 1, n)

# 4. Outcome (Income)
# Depends on X (Education) and U (Confounder)
# True effect of X is 10
Y = 1000 + 10 * X + 50 * U + np.random.normal(0, 5, n)

df = pd.DataFrame({'Income': Y, 'Education': X, 'Distance': Z, 'Ability': U})
df.head()

## 2. Naive OLS (Biased)

If we run a standard regression of Income on Education, we ignore Ability (since it's unobserved in real life).

Because Ability positively boosts both Education and Income, OLS will attribute Ability's effect to Education, resulting in an **upward bias**.

In [ ]:
X_ols = sm.add_constant(df['Education'])
ols_model = sm.OLS(df['Income'], X_ols).fit()
print(ols_model.summary().tables[1])

print(f"\nEstimated Effect: {ols_model.params['Education']:.2f}")
print(f"True Effect: 10.0")

## 3. Instrumental Variable Estimation (2SLS)

We use `linearmodels` to perform Two-Stage Least Squares (2SLS).

- **Dependent**: Income
- **Exog**: Constant (and other controls if we had them)
- **Endog**: Education (The variable correlated with error)
- **Instruments**: Distance (The variable clearing up the mess)

In [ ]:
df['const'] = 1

# IV2SLS(Dependent, Exogenous, Endogenous, Instruments)
iv_model = IV2SLS(df['Income'], df[['const']], df['Education'], df['Distance']).fit()

print(iv_model.summary)

print(f"\nIV Estimated Effect: {iv_model.params['Education']:.2f}")
print(f"True Effect: 10.0")

## 4. Conclusion

As we can see, the OLS estimate was biased (likely > 10) because it conflated Ability with Education. The IV estimate, while often having larger standard errors (less precise), is **consistent** and much closer to the true causal effect of 10.